# 25 — ABSA database check

Inspect everything the ABSA pipeline has written to `data/hotel_reviews.db`:
labeling progress, aspect/sentiment distributions, evidence quality, coined
sub_aspects, roll-ups, gold tables, and a browsable sample of labeled reviews.

All cells are **read-only**.

> ⚠️ DuckDB is single-writer: even a read-only kernel connection blocks
> `absa_label_retrieve.py` from taking the write lock. **Close/restart this
> kernel (or run the last cell) before retrieving a batch.**

In [1]:
import sys
sys.path.insert(0, "../src")

import duckdb
import pandas as pd

from absa_label import DB_PATH

pd.set_option("display.max_colwidth", 120)
con = duckdb.connect(str(DB_PATH), read_only=True)

con.execute("""
    SELECT table_name, table_type FROM information_schema.tables
    WHERE table_name LIKE 'ABSA%' OR table_name LIKE 'REVIEW_ASPECT%'
       OR table_name LIKE 'GOLD%' OR table_name = 'SENTENCE_LABELS'
    ORDER BY 1
""").df()

,table_name,table_type
0,ABSA_SAMPLE,BASE TABLE
1,GOLD_REVIEW_ASPECTS,BASE TABLE
2,GOLD_TRIPADVISOR,BASE TABLE
3,REVIEW_ASPECTS,BASE TABLE
4,REVIEW_ASPECT_ROLLUP,BASE TABLE


## 1. Labeling progress — how far through the 15k sample?

In [2]:
con.execute("""
    SELECT s.split, s.language, s.source,
           count(*)                          AS sampled,
           count(a.review_id)                AS labeled,
           round(count(a.review_id) * 100.0 / count(*), 1) AS pct
    FROM ABSA_SAMPLE s
    LEFT JOIN (SELECT DISTINCT review_id FROM REVIEW_ASPECTS) a USING (review_id)
    GROUP BY 1, 2, 3 ORDER BY 1, 2, 3
""").df()

,split,language,source,sampled,labeled,pct
0,test,en,agoda,286,285,99.7
1,test,en,googlemaps,234,230,98.3
2,test,vi,agoda,199,198,99.5
3,test,vi,googlemaps,793,789,99.5
4,train,en,agoda,2254,2246,99.6
5,train,en,googlemaps,1954,1937,99.1
6,train,vi,agoda,1506,1506,100.0
7,train,vi,googlemaps,6259,6184,98.8
8,val,en,agoda,269,268,99.6
9,val,en,googlemaps,263,263,100.0


In [3]:
# Overall + what the next wave will pick up
con.execute("""
    SELECT count(*) AS sampled,
           count(a.review_id) AS labeled,
           count(*) - count(a.review_id) AS remaining_for_next_waves
    FROM ABSA_SAMPLE s
    LEFT JOIN (SELECT DISTINCT review_id FROM REVIEW_ASPECTS) a USING (review_id)
""").df()

,sampled,labeled,remaining_for_next_waves
0,15000,14880,120


## 2. Aspect × sentiment matrix (the core result)

In [4]:
con.execute("""
    PIVOT (SELECT key_aspect, sentiment FROM REVIEW_ASPECTS)
    ON sentiment IN ('positive', 'neutral', 'negative')
    USING count(*) GROUP BY key_aspect ORDER BY key_aspect
""").df()

,key_aspect,positive,neutral,negative
0,amenity,6755,282,1964
1,experience,8921,420,2564
2,facility,6395,359,5852
3,loyalty,2456,10,609
4,other,112,8,45
5,service,11213,405,3605


In [5]:
# Evidence quality: substring-validity rate per aspect (hallucination check)
con.execute("""
    SELECT key_aspect, count(*) AS rows,
           sum(CASE WHEN NOT evidence_valid THEN 1 ELSE 0 END) AS invalid,
           round(sum(CASE WHEN NOT evidence_valid THEN 1 ELSE 0 END) * 100.0
                 / count(*), 2) AS invalid_pct
    FROM REVIEW_ASPECTS GROUP BY 1 ORDER BY 2 DESC
""").df()

,key_aspect,rows,invalid,invalid_pct
0,service,15223,62.0,0.41
1,facility,12606,52.0,0.41
2,experience,11905,45.0,0.38
3,amenity,9001,35.0,0.39
4,loyalty,3075,8.0,0.26
5,other,165,2.0,1.21


## 3. Sub-aspect vocabulary — seed usage + coined strings

In [6]:
# Top sub_aspects overall
con.execute("""
    SELECT key_aspect, sub_aspect, count(*) AS n
    FROM REVIEW_ASPECTS GROUP BY 1, 2 ORDER BY n DESC LIMIT 20
""").df()

,key_aspect,sub_aspect,n
0,service,staff_attitude,6724
1,amenity,local_convenience,3721
2,experience,overall_satisfaction,3508
3,facility,facility_cleanliness,3338
4,service,food_breakfast,3300
5,facility,room_features,2980
6,experience,price_value,2406
7,facility,technical_equipment,1996
8,amenity,leisure_facilities,1976
9,service,front_desk_service,1771


In [7]:
# Every coined sub_aspect (key_aspect='other') with an example evidence -
# candidates for promoting into the canonical vocabulary later
con.execute("""
    SELECT sub_aspect, count(*) AS n,
           any_value(sentiment) AS example_sentiment,
           any_value(evidence)  AS example_evidence
    FROM REVIEW_ASPECTS WHERE key_aspect = 'other'
    GROUP BY 1 ORDER BY n DESC
""").df()

,sub_aspect,n,example_sentiment,example_evidence
0,food_quality,66,positive,Dùng buổi trưa tại đây rất ngon
1,location,11,neutral,the location was decent
2,beverage_quality,9,positive,"cà phê và các món nước pha chế, uống rất dễ"
3,spa_service,8,positive,buổi massage
4,smoking_policy,5,neutral,đây là khách sạn không hút thuốc trong phòng
...,...,...,...,...
57,staff_compensation,1,negative,trả lương nhân viên hơi thấp
58,pest_issue,1,negative,Mùa mưa hok tránh khỏi muỗi
59,smoking_area,1,negative,There are no smoking room inside the hotel and have to go out from hotel to take a cigarette
60,management_attitude,1,negative,the management doesn't care at all


## 4. Browse labeled reviews — full text + every extracted aspect

In [8]:
N_BROWSE = 5  # rerun the cell for a new random draw

sample_ids = [r[0] for r in con.execute("""
    SELECT DISTINCT review_id FROM REVIEW_ASPECTS ORDER BY random() LIMIT ?
""", [N_BROWSE]).fetchall()]

for rid in sample_ids:
    hotel, lang, text = con.execute(
        "SELECT hotel_name, language, review_text FROM REVIEW_DATA WHERE review_id = ?",
        [rid]).fetchone()
    print(f"=== [{lang}] {hotel}  ({rid}) ===")
    print(" ", " ".join(text.split())[:300])
    for k, sub, sen, ev, ok in con.execute("""
        SELECT key_aspect, sub_aspect, sentiment, evidence, evidence_valid
        FROM REVIEW_ASPECTS WHERE review_id = ? ORDER BY aspect_rank
    """, [rid]).fetchall():
        flag = "" if ok else "  << INVALID EVIDENCE"
        print(f"    {k:11} <- {sub:26} {sen:8} | {ev[:70]!r}{flag}")
    print()

=== [en] Tulip Hotel 2  (ChdDSUhNMG9nS0VJQ0FnSUROdkpMdHZnRRAB) ===
  This hotel is a bit dated now but the rooms are a generous size with lots of storage. En-suite are acceptable. The central location is good for getting around but the noise of the town can keep you awake. The hotel does not offer parking, …
    facility    <- facility_condition         negative | 'This hotel is a bit dated now'
    facility    <- room_features              positive | 'the rooms are a generous size with lots of storage'
    facility    <- bathroom_facilities        neutral  | 'En-suite are acceptable'
    amenity     <- local_convenience          positive | 'The central location is good for getting around'
    experience  <- noise_quietness            negative | 'the noise of the town can keep you awake'
    amenity     <- parking_facility           negative | 'The hotel does not offer parking'

=== [vi] Chloe Gallery  (ChZDSUhNMG9nS0VJQ0FnSURoNG9xLWFREAE) ===
  Soup có mùi khét. Toilet chật, phải chờ 

## 5. Roll-up sanity — per-review macro sentiment vs overall rating

In [9]:
con.execute("SELECT * FROM REVIEW_ASPECT_ROLLUP LIMIT 10").df()

,review_id,asp5_facility,asp5_amenity,asp5_service,asp5_experience,asp5_loyalty,n_aspects,label_model,labeled_at
0,1c845bbb-5204-4e38-805b-b716c6e6bba8,NaN,negative,positive,NaN,None,2,claude-sonnet-5,2026-07-17 14:21:33.586496
1,1c8c73d6-f5ac-4275-bc5f-dca62daff2ec,NaN,NaN,positive,positive,None,2,claude-sonnet-5,2026-07-17 14:21:33.591246
2,1caaf908-44cf-428b-af90-3b127cec03d9,positive,negative,negative,positive,None,6,claude-sonnet-5,2026-07-17 14:21:33.602939
3,1cb26f4b-6773-4e38-8638-1ae43d8862e6,positive,negative,positive,positive,None,5,claude-sonnet-5,2026-07-17 14:21:33.612367
4,1cd449ba-35d1-47ba-a509-e0ddd5939179,positive,positive,positive,NaN,None,3,claude-sonnet-5,2026-07-17 14:21:33.619415
5,1cedb4bd-9663-438d-b9ec-20a39628de11,positive,positive,NaN,NaN,None,2,claude-sonnet-5,2026-07-17 14:21:33.623883
6,1cf640ef-4bff-43f4-80f0-3fe1a0db63ad,negative,positive,negative,NaN,None,3,claude-sonnet-5,2026-07-17 14:21:33.630455
7,1cf8addd-094f-40b0-a92b-99b7e3187ebd,positive,negative,NaN,positive,None,3,claude-sonnet-5,2026-07-17 14:21:33.636851
8,1d0e681d-e5fa-419f-a57b-6624ab96bf7e,positive,positive,NaN,positive,None,6,claude-sonnet-5,2026-07-17 14:21:33.648729
9,1d1928bc-206f-437b-9005-e9195cbb6959,NaN,negative,NaN,NaN,None,1,claude-sonnet-5,2026-07-17 14:21:33.652125


In [10]:
# Weak sanity signal: reviews the model calls facility-negative should have a
# lower overall reviewer rating than facility-positive ones
con.execute("""
    SELECT r.asp5_facility AS facility_sentiment,
           count(*) AS n,
           round(avg(d.rating_normalized), 2) AS avg_reviewer_rating_of5
    FROM REVIEW_ASPECT_ROLLUP r JOIN REVIEW_DATA d USING (review_id)
    WHERE r.asp5_facility IS NOT NULL AND d.rating_normalized IS NOT NULL
    GROUP BY 1 ORDER BY 3 DESC
""").df()

,facility_sentiment,n,avg_reviewer_rating_of5
0,positive,3826,4.65
1,neutral,758,4.05
2,negative,2806,2.94


## 6. Silver vs gold — quick agreement preview (GMap star tags)

For reviews that have BOTH a silver roll-up and a reviewer star tag, how often
do they agree? (Full evaluation comes later; this is the early temperature check.)

In [11]:
con.execute("""
    WITH pairs AS (
        SELECT g.review_id, g.key_aspect, g.sentiment AS gold,
               CASE g.key_aspect
                    WHEN 'facility' THEN r.asp5_facility
                    WHEN 'service'  THEN r.asp5_service
                    WHEN 'amenity'  THEN r.asp5_amenity END AS silver
        FROM GOLD_REVIEW_ASPECTS g
        JOIN REVIEW_ASPECT_ROLLUP r USING (review_id)
        WHERE g.gold_source = 'gmap_tag'
    )
    SELECT key_aspect,
           count(*) FILTER (WHERE silver IS NOT NULL)        AS both_present,
           round(avg(CASE WHEN silver = gold THEN 1.0 ELSE 0.0 END)
                 FILTER (WHERE silver IS NOT NULL) * 100, 1) AS agreement_pct
    FROM pairs GROUP BY 1 ORDER BY 1
""").df()

,key_aspect,both_present,agreement_pct
0,amenity,1447,80.1
1,facility,1559,69.1
2,service,2571,88.8


## 7. Gold tables + sentence bridge status

In [12]:
# Gold + bridge tables (tolerant of tables not built yet)
existing = {r[0] for r in con.execute("""
    SELECT table_name FROM information_schema.tables
""").fetchall()}

parts = []
if "GOLD_REVIEW_ASPECTS" in existing:
    parts.append("SELECT 'GOLD_REVIEW_ASPECTS' AS tbl, gold_source AS detail, count(*) AS rows FROM GOLD_REVIEW_ASPECTS GROUP BY 2")
if "GOLD_TRIPADVISOR" in existing:
    parts.append("SELECT 'GOLD_TRIPADVISOR', 'spans', count(*) FROM GOLD_TRIPADVISOR")
if "SENTENCE_LABELS" in existing:
    parts.append("SELECT 'SENTENCE_LABELS', 'rows (built by absa_bridge.py)', count(*) FROM SENTENCE_LABELS")
else:
    print("SENTENCE_LABELS not built yet - run src/absa_bridge.py after labeling")

con.execute(" UNION ALL ".join(parts) + " ORDER BY 1, 2").df()


SENTENCE_LABELS not built yet - run src/absa_bridge.py after labeling


,tbl,detail,rows
0,GOLD_REVIEW_ASPECTS,gmap_highlight,31634
1,GOLD_REVIEW_ASPECTS,gmap_tag,94968
2,GOLD_TRIPADVISOR,spans,54326


In [13]:
con.close()
print("connection closed - safe to run retrieve now")

connection closed - safe to run retrieve now
